# Tutorial 1: primeiros passos com ObsPy usando a estação ISPT

**Para quem é:** quem está começando com Python científico e sismologia. Nenhuma biblioteca é considerada conhecida.

**Objetivo:** aprender a ler, visualizar e transformar sinais sísmicos com a biblioteca **ObsPy**, usando dados reais da estação **EC.ISPT** (Isla de la Plata, Equador, 2017). No final, rodamos os modelos do **E3WS** num terremoto real da ISPT.

### Como usar este notebook
1. Abra no VS Code e escolha o kernel **Python (E3WS)** (canto superior direito, "Select Kernel").
2. Rode **uma célula por vez** com `Shift+Enter`.
3. Em cada parte: leia o **conceito**, rode o **código** (cada linha tem um comentário `#` explicando o que faz), leia a tabela **"Funções desta célula"** e depois o **"O que observar"**.
4. Faça os **exercícios**: é mexendo nos números que se aprende. Os arquivos originais nunca são alterados.

### Como ler o código
- Tudo depois de `#` é **comentário**: o Python ignora, é só explicação.
- `variavel = algo` guarda um valor com um nome, para usar depois.
- `objeto.funcao(...)` chama uma função que "pertence" ao objeto. Ex.: `tr.filter(...)` filtra o Trace `tr`.
- `parametro=valor` dentro dos parênteses é um **parâmetro nomeado**: um ajuste da função.

### Roteiro

| Parte | Assunto | Ligação com o E3WS |
|---|---|---|
| 0 | Preparação | |
| 1 | O que é um sinal sísmico | Seção 2 do artigo |
| 2 | Ler um arquivo: `Stream` e `Trace` | |
| 3 | Lacunas e sobreposições | Qualidade dos dados |
| 4 | Ver um dia inteiro | |
| 5 | Recortar um evento | Janelas de 10 s |
| 6 | Pré-processamento | Início da função `st_FV` |
| 7 | **Resposta instrumental: de counts para m/s²** | Seção 2 (deconvolução) |
| 8 | Domínio da frequência | Atributos espectrais |
| 9 | Filtros | 1–7 Hz e 1–45 Hz |
| 10 | Ondas P e S, distância | Picker e distância |
| 11 | Envelope | Atributos temporais |
| 12 | STA/LTA, o detector clássico | Por que o E3WS usa ML |
| 13 | Três componentes e polarização | Back-azimuth |
| 14 | Os 140 atributos do E3WS | Seção 3.4 |
| 15 | **Primeira estimativa do E3WS na ISPT** | Seção 3.3 |

---
## Parte 0: preparação

**Conceito.** Um programa em Python começa "trazendo" as bibliotecas que vai usar (`import`). Depois definimos onde estão os arquivos.

- **NumPy** (`np`): contas com vetores. Um sinal sísmico é um vetor de números.
- **Matplotlib** (`plt`): gráficos.
- **ObsPy**: tudo de sismologia (ler, filtrar, remover resposta, detectar).

Os dados estão no padrão **SDS** (*SeisComP Data Structure*): uma pasta por canal e **um arquivo por dia**.

In [ ]:
import numpy as np                      # biblioteca de contas com vetores; "np" é o apelido
import matplotlib.pyplot as plt         # biblioteca de gráficos; "plt" é o apelido
import obspy                            # biblioteca de sismologia
from obspy import read, UTCDateTime     # traz duas funções da ObsPy para usar pelo nome curto

plt.rcParams["figure.figsize"] = (12, 4)            # tamanho padrão dos gráficos: 12 x 4 polegadas
print("ObsPy", obspy.__version__)                   # mostra a versão instalada da ObsPy
print("NumPy", np.__version__)                      # mostra a versão do NumPy

# Pastas no Google Drive (o "r" antes das aspas faz o Python aceitar a barra invertida "\")
DADOS = r"G:\Meu Drive\Mestrado\PPGEEC-UFC\dataset\ISPT_SEISMIC\2017\EC\ISPT"   # formas de onda
PASTA_PZ = r"G:\Meu Drive\Mestrado\PPGEEC-UFC\dataset\ISPT_SEISMIC\pz"          # resposta instrumental

def caminho(componente, dia_juliano, ano=2017):     # cria uma função: recebe componente e dia
    canal = "HH" + componente                       # junta textos: "HH" + "Z" vira "HHZ"
    nome = f"EC.ISPT..{canal}.D.{ano}.{dia_juliano:03d}"   # nome do arquivo; :03d escreve o dia com 3 dígitos
    return rf"{DADOS}\{canal}.D\{nome}"             # devolve o caminho completo

print(caminho("Z", 180))                            # testa a função: arquivo do vertical, dia 180

#### Funções desta célula

| Função / comando | O que faz | Parâmetros usados |
|---|---|---|
| `import x as y` | Carrega a biblioteca `x` com o apelido `y` | — |
| `from x import a, b` | Traz só as funções `a` e `b` da biblioteca `x` | — |
| `plt.rcParams[...]` | Muda uma configuração padrão do Matplotlib | `"figure.figsize"`: (largura, altura) em polegadas |
| `print(...)` | Mostra algo na tela | qualquer valor, separado por vírgulas |
| `def nome(...):` | Cria uma função sua | `componente`, `dia_juliano` e `ano=2017` (valor padrão, usado se você não informar) |
| `f"...{x}..."` | Texto com valores embutidos (*f-string*) | `{dia:03d}`: inteiro com 3 dígitos, completando com zeros (5 → `005`) |
| `return` | Devolve o resultado da função | — |

**O que observar:** o caminho impresso deve existir no seu Google Drive. Se der erro mais adiante dizendo que o arquivo não existe, confira a variável `DADOS`.

---
## Parte 1: o que é um sinal sísmico

**Conceito.** Um **sismômetro** mede o movimento do solo e gera uma tensão elétrica. Um **digitalizador** transforma essa tensão em números, várias vezes por segundo.

- **Taxa de amostragem:** quantos números por segundo. Na ISPT são **100 por segundo (100 Hz)**, a mesma taxa do E3WS. Então 3 s de onda P = 300 números.
- **Amostra:** cada número. A unidade é **counts** (contagens do digitalizador). Para virar unidade física (m/s, m/s²) é preciso a **resposta instrumental** (Parte 7).
- **Três componentes:** **Z** (vertical), **N** (norte–sul) e **E** (leste–oeste).

**Nome do canal (padrão SEED):** `EC.ISPT.00.HHZ` = `REDE.ESTAÇÃO.LOCALIZAÇÃO.CANAL`

| Parte | Valor | Significado |
|---|---|---|
| Rede | `EC` | Rede Sísmica do Equador (operada pelo IG-EPN, Quito) |
| Estação | `ISPT` | Isla de la Plata |
| Localização | `00` | Distingue sensores no mesmo local (atenção a isso na Parte 7) |
| Canal, 1ª letra | `H` | Alta taxa de amostragem (80 a 250 Hz) |
| Canal, 2ª letra | `H` | Sismômetro de alto ganho (mede velocidade) |
| Canal, 3ª letra | `Z` | Componente vertical |

**Dia juliano:** os arquivos usam o número do dia no ano (1 a 365) em vez de dia/mês.

In [ ]:
d = UTCDateTime(year=2017, julday=180)       # cria uma data a partir do ano e do dia juliano
print("Dia 180 de 2017 =", d.date)           # .date mostra só a data (sem a hora)

d2 = UTCDateTime("2017-06-29")               # cria uma data a partir de um texto no formato ano-mês-dia
print("29/06/2017 = dia", d2.julday)         # .julday devolve o dia juliano

primeiro = UTCDateTime(year=2017, julday=153)   # primeiro dia com dados
ultimo = UTCDateTime(year=2017, julday=244)     # último dia com dados
print("Dados de", primeiro.date, "até", ultimo.date)

#### Funções desta célula

| Função / atributo | O que faz | Parâmetros usados |
|---|---|---|
| `UTCDateTime(...)` | Cria um instante de tempo em UTC (horário universal, usado em toda a sismologia) | `year`, `julday`; ou um texto como `"2017-06-29T18:53:03.5"` |
| `.date` | Só a data do instante | — |
| `.julday` | O dia juliano do instante | — |

Dá para fazer contas com `UTCDateTime`: `d + 60` é 60 segundos depois; `d2 - d` dá a diferença em segundos.

**Exercício 1:** qual dia juliano corresponde a 15/07/2017? Qual data é o dia 220?

---
## Parte 2: ler um arquivo, `Stream` e `Trace`

**Conceito.** A função `read()` devolve um **Stream**, que é uma lista de **Traces**.
- **Trace:** um trecho **contínuo** de sinal de um canal. Tem `data` (os números) e `stats` (os metadados).
- Se o arquivo tem um buraco no meio, o Stream vem com **mais de um Trace**.

Analogia: o Stream é uma pasta; cada Trace é uma folha com um pedaço contínuo da gravação.

In [ ]:
st = read(caminho("Z", 180))      # lê o arquivo do vertical do dia 180; "st" é o Stream
print(st)                         # mostra o resumo: quantos Traces e o período de cada um

In [ ]:
tr = st[0]                        # pega o primeiro Trace da lista (em Python, a contagem começa em 0)
print(tr.stats)                   # mostra todos os metadados do Trace

In [ ]:
print("Primeiros 10 valores:", tr.data[:10])     # [:10] = do início até o 10º elemento
print("Tipo dos dados:", tr.data.dtype)          # int32 = números inteiros de 32 bits
print("Número de amostras:", tr.stats.npts)      # npts = number of points
duracao_h = tr.stats.npts / tr.stats.sampling_rate / 3600   # amostras ÷ (amostras/s) ÷ 3600 = horas
print("Duração (h): %.2f" % duracao_h)           # %.2f escreve o número com 2 casas decimais
print("Média: %.1f" % tr.data.mean())            # média de todos os valores
print("Mínimo: %d | Máximo: %d" % (tr.data.min(), tr.data.max()))   # %d escreve um inteiro

#### Funções desta célula

| Função / atributo | O que faz | Parâmetros usados |
|---|---|---|
| `read(arquivo)` | Lê um arquivo de forma de onda (miniSEED, SAC etc.) e devolve um Stream | o caminho do arquivo. Também aceita `starttime=` e `endtime=` (Parte 5) |
| `st[0]` | Pega um Trace do Stream pela posição | `0` = primeiro, `-1` = último |
| `tr.stats` | Metadados: `network`, `station`, `location`, `channel`, `starttime`, `endtime`, `sampling_rate` (Hz), `delta` (s entre amostras), `npts` (nº de amostras) | — |
| `tr.data` | O sinal em si, um vetor NumPy | — |
| `vetor[:10]` | Fatia: os 10 primeiros elementos | `[a:b]` = do índice `a` até antes de `b` |
| `.mean()`, `.min()`, `.max()` | Média, mínimo e máximo de um vetor NumPy | — |
| `"%.2f" % x` | Formata um número como texto | `%.2f` = 2 casas decimais; `%d` = inteiro |

**O que observar**
- O dia 180 vem com **2 Traces**: houve uma interrupção pouco antes das 19h.
- A **média não é zero** (cerca de +277 counts neste dia; no dia 153 é cerca de −2400). Isso é um **offset** do equipamento, não movimento do solo, e muda de um dia para outro. Vamos removê-lo na Parte 6.

**Exercício 2:** leia o componente `N` do mesmo dia e compare a média e o desvio padrão (`tr.data.std()`) com os do `Z`.

---
## Parte 3: lacunas (gaps) e sobreposições (overlaps)

**Conceito.** Dados reais têm falhas: queda de energia, perda de comunicação, reinício do equipamento.
- **Gap:** falta um trecho.
- **Overlap:** dois Traces cobrem o mesmo intervalo de tempo.

O dia 153 (primeiro dia da coleta) é um bom exemplo de dado "bagunçado".

In [ ]:
st180 = read(caminho("Z", 180))      # lê o dia 180
st180.print_gaps()                   # lista as falhas: início, fim e duração

In [ ]:
st153 = read(caminho("Z", 153))      # lê o dia 153 (primeiro dia da coleta)
print(st153)                         # resumo: repare na quantidade de Traces
st153.print_gaps()                   # lista falhas e sobreposições

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `st.print_gaps()` | Mostra uma tabela com cada falha: último instante antes, próximo instante depois, duração (`Delta`, em s) e nº de amostras. Delta **positivo** = gap; **negativo** = overlap | — |

**O que observar:** o dia 180 tem 1 gap de ~101 s. O dia 153 tem **16 Traces**, quase todos em pares que se sobrepõem.

Para ter um sinal contínuo, usamos `merge()`.

In [ ]:
st153_unido = st153.copy()           # faz uma cópia, para não alterar o original
st153_unido.merge(method=1)          # junta os Traces num só, resolvendo as sobreposições
print(st153_unido)                   # agora é 1 Trace; "(masked)" indica que há trechos sem dado

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `st.copy()` | Cria uma cópia independente. Muitas funções da ObsPy **alteram o próprio objeto**, então copiar protege o original | — |
| `st.merge(...)` | Junta Traces do mesmo canal num só | `method=1`: nas sobreposições, usa os dados de forma inteligente (interpolando). `fill_value=0` (opcional): preenche os gaps com zeros em vez de deixá-los "mascarados" |

**Cuidado com `fill_value=0`:** zero é um valor inventado. Um filtro aplicado sobre um trecho de zeros gera artefatos nas bordas. Use só quando necessário (como no dayplot da Parte 4).

**Exercício 3:** quantos gaps têm os dias 154 e 200? Dica: `len(read(caminho("Z", 154)).get_gaps())`.

---
## Parte 4: ver um dia inteiro

**Conceito.** O `dayplot` desenha um dia em várias linhas, uma por hora, como os antigos sismógrafos de papel. Os eventos aparecem como "riscos" maiores.

Antes, filtramos acima de 1 Hz para tirar a oscilação lenta causada pelo oceano (Parte 9 explica). A célula pode levar alguns segundos.

In [ ]:
dia = read(caminho("Z", 180))        # lê o dia inteiro
dia.merge(fill_value=0)              # junta tudo num Trace, preenchendo o gap com zeros
dia.detrend("demean")                # tira a média (o offset)
dia.filter("highpass", freq=1.0)     # mantém só frequências acima de 1 Hz
dia.plot(type="dayplot",             # tipo de gráfico: um dia em linhas
         interval=60,                # cada linha mostra 60 minutos
         vertical_scaling_range=2000,    # amplitude (em counts) que cabe em cada linha
         one_tick_per_line=True,     # um rótulo de horário por linha
         color=["k", "r", "b", "g"], # cores que se alternam: preto, vermelho, azul, verde
         size=(1000, 1200))          # tamanho da figura em pixels (largura, altura)

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `st.detrend("demean")` | Subtrai a média | `"demean"`: média; `"linear"`: reta (Parte 6) |
| `st.filter("highpass", ...)` | Filtro passa-alta | `freq`: frequência de corte em Hz |
| `st.plot(...)` | Desenha o Stream | `type="dayplot"`; `interval` (min por linha); `vertical_scaling_range` (amplitude por linha: menor = riscos maiores); `one_tick_per_line`; `color`; `size` |

**O que observar:** procure a linha das 18h. Perto das 18:53 está o **maior evento do dia**: é o terremoto que vamos estudar. Há outro forte por volta das 04:23.

O rótulo do eixo diz "local time = UTC − 03:00" porque a ObsPy usa o fuso do seu computador. Ignore: as horas nas linhas estão em **UTC**.

**Exercício 4:** faça o dayplot do dia 220. Em que horários aparecem os maiores eventos? Teste `vertical_scaling_range=500`.

---
## Parte 5: recortar um evento

**Conceito.** Não precisamos carregar o dia inteiro para estudar um evento: `read()` aceita início e fim e lê só aquele trecho.

Vamos ler os **3 componentes** do evento de 29/06/2017, de 18:52:30 a 18:54:30 (2 minutos).

In [ ]:
inicio = UTCDateTime("2017-06-29T18:52:30")    # início do recorte
fim = UTCDateTime("2017-06-29T18:54:30")       # fim do recorte

evento = obspy.Stream()                        # cria um Stream vazio
for comp in ["E", "N", "Z"]:                   # repete o bloco abaixo para cada componente
    evento += read(caminho(comp, 180),         # lê o arquivo do componente...
                   starttime=inicio,           # ...a partir deste instante...
                   endtime=fim)                # ...até este. "+=" acrescenta ao Stream
evento.merge()                                 # garante um Trace por componente
print(evento)                                  # deve mostrar 3 Traces de 120 s

In [ ]:
evento.plot()                                  # desenha os 3 componentes, um embaixo do outro

#### Funções desta célula

| Função / comando | O que faz | Parâmetros usados |
|---|---|---|
| `obspy.Stream()` | Cria um Stream vazio | — |
| `for x in lista:` | Repete o bloco indentado para cada item da lista | — |
| `read(..., starttime=, endtime=)` | Lê só o intervalo pedido | `starttime`, `endtime`: `UTCDateTime` |
| `st += outro` | Acrescenta os Traces de `outro` em `st` | — |
| `st.plot()` | Gráfico padrão: um painel por Trace | (opcional) `starttime`, `endtime`, `outfile="figura.png"` para salvar |

**O que observar**
- Cada componente "flutua" num nível diferente (o offset).
- Há duas chegadas: uma por volta de 18:53:03 e outra, mais forte nos horizontais, perto de 18:53:14. São as ondas **P** e **S** (Parte 10).

In [ ]:
t1 = UTCDateTime("2017-06-29T18:52:58")        # início do zoom
t2 = UTCDateTime("2017-06-29T18:53:20")        # fim do zoom
zoom = evento.slice(t1, t2)                    # recorta, devolvendo um novo Stream
zoom.plot()                                    # desenha só o trecho recortado

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `st.slice(t1, t2)` | Devolve um **novo** Stream com o trecho; o original não muda | `t1`, `t2`: `UTCDateTime` |
| `st.trim(t1, t2)` | Mesmo recorte, mas **no próprio objeto** (o original é cortado) | idem |

**Exercício 5:** recorte os 10 s antes do evento (só ruído) e compare a amplitude com a do evento.

---
## Parte 6: pré-processamento (o que o E3WS faz primeiro)

**Conceito.** Antes de qualquer análise, o sinal é "limpo". São os mesmos passos da Seção 2 do artigo e do início da função `st_FV` do E3WS (`real_time/functions/pb_utils_v16.py`, linhas 27 a 29):

1. **Tirar a média** (`demean`): o sinal passa a oscilar em torno de zero.
2. **Tirar a tendência** (`linear`): remove uma deriva lenta, ajustando e subtraindo uma reta.
3. **Taper:** suaviza as pontas até zero. Sem isso, o corte brusco da janela gera artefatos na FFT e nos filtros.

In [ ]:
z_bruto = evento.select(component="Z")[0].copy()   # pega o Trace do vertical e copia

z_demean = z_bruto.copy()                     # cópia para o passo 1
z_demean.detrend("demean")                    # passo 1: tira a média

z_detrend = z_demean.copy()                   # cópia para o passo 2
z_detrend.detrend("linear")                   # passo 2: tira a tendência linear

z_taper = z_detrend.copy()                    # cópia para o passo 3
z_taper.taper(max_percentage=0.05,            # passo 3: suaviza 5% de cada ponta...
              type="cosine")                  # ...com formato de cosseno

etapas = [z_bruto, z_demean, z_detrend, z_taper]                      # lista com as 4 versões
nomes = ["1. Bruto (counts)", "2. Sem média", "3. Sem tendência", "4. Com taper"]

fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)   # figura com 4 painéis empilhados
for a, t, nome in zip(ax, etapas, nomes):     # percorre painel, versão e nome ao mesmo tempo
    a.plot(t.times(), t.data, "k", lw=0.5)    # desenha: x = tempo (s), y = sinal; preto, linha fina
    a.set_title(nome, loc="left")             # título à esquerda do painel
    a.axhline(0, color="r", lw=0.8)           # linha horizontal vermelha no zero
ax[-1].set_xlabel("Tempo desde 18:52:30 (s)") # rótulo do eixo x só no último painel
plt.tight_layout()                            # ajusta espaçamentos para nada ficar sobreposto

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `st.select(component="Z")` | Filtra o Stream pelos metadados | `component`, `channel`, `station`... |
| `tr.detrend(tipo)` | Remove média ou tendência | `"demean"`: média; `"linear"`: reta entre o primeiro e o último ponto; `"simple"`: reta ligando as pontas |
| `tr.taper(...)` | Multiplica as pontas por uma curva que vai a zero | `max_percentage`: fração **de cada ponta** (0,05 = 5%); `type`: formato (`"cosine"`, `"hann"`...) |
| `plt.subplots(l, c, ...)` | Cria figura com `l` linhas e `c` colunas de painéis | `figsize`; `sharex=True`: mesmo eixo x |
| `zip(a, b, c)` | Percorre várias listas juntas | — |
| `tr.times()` | Vetor com o tempo de cada amostra, em segundos desde o início | — |
| `ax.plot(x, y, ...)` | Desenha uma linha | `"k"`: cor preta; `lw`: espessura |
| `ax.axhline(y)` | Linha horizontal | `color`, `lw` |
| `ax[-1]` | O último painel | — |

**O que observar:** o painel 1 está longe da linha vermelha (zero); o painel 2 oscila em torno dela; no painel 4 as pontas vão suavemente a zero.

**Reamostragem:** o E3WS trabalha a 100 Hz, e a ISPT já grava a 100 Hz. Com outra taxa, seria `tr.resample(100)`.

**Exercício 6:** use `max_percentage=0.3` e veja como um taper exagerado "come" o sinal.

---
## Parte 7: resposta instrumental, de counts para m/s²

**Conceito.** O arquivo guarda **counts**, a saída do equipamento. Cada sensor e digitalizador transforma o movimento do solo de um jeito próprio: tem um **ganho** (quantos counts por m/s) e responde de forma diferente a cada frequência. Essa "assinatura" é a **resposta instrumental**.

**Remover a resposta** (deconvolução) é fazer a conta ao contrário: dos counts, recuperar o movimento real do solo em unidades físicas. É obrigatório para o E3WS, cujos modelos foram treinados com **aceleração em m/s²**.

A resposta da ISPT está num arquivo **StationXML** (`ISPT.xml`), o formato padrão de metadados de estações.

In [ ]:
from obspy import read_inventory                      # função que lê arquivos de metadados

inv = read_inventory(PASTA_PZ + r"\ISPT.xml")         # lê o StationXML da ISPT
print(inv)                                            # resumo: redes, estações e canais

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `read_inventory(arquivo)` | Lê metadados de estações (StationXML, RESP, dataless SEED) e devolve um **Inventory** | caminho do arquivo |

**O que observar:** o arquivo tem **96 canais**, porque guarda toda a história da estação desde 2008. Cada troca de equipamento gera uma nova "época" (*epoch*) com uma resposta diferente. Precisamos achar a época válida em 2017.

In [ ]:
t_evento = UTCDateTime("2017-06-29T18:53:00")          # um instante dentro dos nossos dados

inv_2017 = inv.select(channel="HH?",                   # só canais HH (o "?" vale qualquer letra: Z, N, E)
                      time=t_evento)                   # só as épocas válidas neste instante

for canal in inv_2017[0][0]:                           # [0] = 1ª rede; [0] = 1ª estação; percorre os canais
    print(f"Localização '{canal.location_code}' | canal {canal.code} | desde {canal.start_date.date}")
    print("   Sensor:       ", canal.sensor.description)            # modelo do sismômetro
    print("   Digitalizador:", canal.data_logger.description)       # modelo do digitalizador
    print("   Azimute: %s° | Mergulho: %s°" % (canal.azimuth, canal.dip))   # orientação do componente
    sens = canal.response.instrument_sensitivity                    # sensibilidade total
    print("   Sensibilidade: %.4g counts por %s" % (sens.value, sens.input_units))

#### Funções desta célula

| Função / atributo | O que faz | Parâmetros usados |
|---|---|---|
| `inv.select(...)` | Filtra o Inventory | `channel="HH?"` (`?` = um caractere qualquer, `*` = vários); `time`: só épocas ativas nesse instante. Também aceita `network`, `station`, `location` |
| `inv[0][0]` | Inventory → rede → estação. Percorrer a estação dá os canais | — |
| `canal.location_code`, `.code`, `.start_date` | Localização, nome do canal e início da época | — |
| `canal.sensor.description`, `canal.data_logger.description` | Modelo do sensor e do digitalizador | — |
| `canal.azimuth`, `canal.dip` | Orientação: azimute (0° = norte, 90° = leste); mergulho (−90° = vertical para cima) | — |
| `response.instrument_sensitivity.value` | Ganho total na frequência de referência | — |

**O que observar**
- Desde **02/06/2017 19:15**, exatamente quando nossos dados começam, a ISPT tem um sismômetro **Nanometrics Trillium 120PA** (banda larga, mede velocidade) e um digitalizador **Reftek RT-130**.
- Sensibilidade de ~3,78 × 10⁸ **counts por m/s**: 1 count ≈ 2,6 nanômetros por segundo.
- **Armadilha:** no StationXML a localização dessa época é **vazia** (`''`), mas nossos arquivos dizem **`00`**. A ObsPy procura a resposta pelo nome completo e **não vai achar**. Veja:

In [ ]:
teste = evento.copy()                                  # cópia do evento (ainda em counts)
try:                                                   # tenta executar o bloco abaixo...
    teste.remove_response(inventory=inv, output="ACC") # ...remover a resposta sem corrigir nada
except Exception as erro:                              # se der erro, cai aqui em vez de parar o notebook
    print("Erro:", erro)                               # mostra a mensagem de erro

print("Localização nos dados:", repr(evento[0].stats.location))   # repr mostra as aspas, para ver se está vazio

#### Funções desta célula

| Comando | O que faz |
|---|---|
| `try: ... except Exception as erro:` | Tenta executar; se der erro, executa o bloco `except` em vez de interromper. Útil para mostrar um erro sem travar o notebook |
| `repr(x)` | Mostra o valor "cru", com aspas. Assim, `''` (texto vazio) fica visível |

**A correção:** alterar a localização nos dados de `00` para vazio, para combinar com o StationXML. Isso só muda o "rótulo", não o sinal. Vale confirmar com o Pablo que o sensor `00` dos arquivos é o mesmo da época de 2017 (as datas indicam que sim).

In [ ]:
for tr in evento:                     # para cada Trace (E, N, Z) do evento...
    tr.stats.location = ""            # ...troca a localização "00" por vazio

resposta = inv.get_response("EC.ISPT..HHZ", t_evento)   # pega a resposta do vertical neste instante
resposta.plot(min_freq=0.001,         # desenha a resposta a partir de 0,001 Hz...
              output="VEL",           # ...considerando a entrada em velocidade...
              label="ISPT HHZ")       # ...com este nome na legenda

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `inv.get_response(id, tempo)` | Devolve a resposta de um canal num instante | `id`: `"REDE.ESTACAO.LOCAL.CANAL"`; `tempo`: `UTCDateTime` |
| `resposta.plot(...)` | Desenha a **curva de resposta**: em cima, a amplitude (ganho por frequência); embaixo, a fase | `min_freq`: menor frequência do gráfico (Hz); `output`: `"VEL"`, `"ACC"` ou `"DISP"`; `label`: nome na legenda |

**O que observar:** a curva de amplitude é **plana** entre cerca de 0,01 Hz (período de ~100 s) e algumas dezenas de Hz: nessa faixa o sensor registra todas as frequências com o mesmo ganho. Fora dela, o ganho cai. Por isso é um sensor de **banda larga** (o "120" do nome vem dos 120 s de período).

In [ ]:
ev = evento.copy()                    # cópia: "ev" será o evento em unidades físicas
ev.detrend("demean")                  # tira a média
ev.detrend("linear")                  # tira a tendência
ev.taper(max_percentage=0.05)         # suaviza as pontas

ev.remove_response(inventory=inv,     # remove a resposta usando o StationXML...
                   output="ACC",      # ...obtendo aceleração (m/s²)...
                   pre_filt=[0.05, 0.1, 40, 45])   # ...com um filtro de segurança (explicado abaixo)

ev_vel = evento.copy()                # outra cópia, para ver também a velocidade
ev_vel.detrend("demean")              # mesmo pré-processamento
ev_vel.detrend("linear")
ev_vel.taper(max_percentage=0.05)
ev_vel.remove_response(inventory=inv, output="VEL", pre_filt=[0.05, 0.1, 40, 45])   # velocidade (m/s)

for a, v in zip(ev, ev_vel):          # percorre aceleração e velocidade juntas, componente a componente
    pga = np.abs(a.data).max()        # PGA: maior aceleração em módulo (Peak Ground Acceleration)
    pgv = np.abs(v.data).max()        # PGV: maior velocidade em módulo (Peak Ground Velocity)
    print("%s | PGA = %.2e m/s² (%.4f %%g) | PGV = %.2e m/s" % (a.stats.channel, pga, pga / 9.81 * 100, pgv))

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `st.remove_response(...)` | Deconvolução: transforma counts em unidade física | `inventory`: o Inventory com a resposta; `output`: `"DISP"` (m), `"VEL"` (m/s) ou `"ACC"` (m/s²); `pre_filt`: filtro aplicado junto (abaixo). Outros: `water_level=60` (padrão, limita a amplificação de ruído) |
| `pre_filt=[f1, f2, f3, f4]` | Filtro em forma de "trapézio" na frequência: zero abaixo de `f1`, sobe até `f2`, plano até `f3`, cai até zero em `f4`. Evita que a deconvolução amplifique ruído onde o sensor quase não registra. `f4` deve ficar abaixo de 50 Hz (metade da taxa de 100 Hz) | `[0.05, 0.1, 40, 45]` |
| `np.abs(x)` | Valor absoluto (módulo) de cada elemento | — |
| `%.2e` | Notação científica: `9.50e-04` = 0,00095 | — |
| `%%` | Escreve o símbolo `%` dentro de um texto formatado | — |

**O que observar**
- A maior aceleração é ~0,001 m/s² (cerca de **0,01% da gravidade**). É um tremor pequeno, imperceptível para pessoas, mas claríssimo para o sensor.
- De agora em diante, **`ev` está em m/s²**, a mesma unidade do E3WS. As próximas partes usam `ev`.

**Exercício 7:** remova a resposta com `output="DISP"` e calcule o deslocamento máximo. Quantos micrômetros o solo se moveu?

In [ ]:
ev.plot()                             # o evento agora em aceleração (m/s²), já centrado em zero

---
## Parte 8: domínio da frequência

**Conceito.** Um sinal pode ser visto de dois jeitos:
- **No tempo:** como a amplitude varia segundo a segundo.
- **Na frequência:** quanto de cada "nota" (frequência, em Hz) existe no sinal.

Analogia: um acorde de violão. No tempo você vê a onda misturada; na frequência, as notas que o formam.

A **FFT** (Transformada Rápida de Fourier) faz essa conversão. Vamos comparar 25 s de **ruído** (antes do evento) com 25 s de **evento**.

In [ ]:
z = ev.select(component="Z")[0]               # vertical, já em m/s²
t0 = z.stats.starttime                        # instante inicial (18:52:30)

ruido = z.slice(t0 + 5, t0 + 30).copy()       # de 5 a 30 s: só ruído
sinal = z.slice(t0 + 33, t0 + 58).copy()      # de 33 a 58 s: o evento
ruido.taper(0.05)                             # suaviza as pontas antes da FFT
sinal.taper(0.05)

def espectro(tr):                             # função: recebe um Trace, devolve frequências e amplitudes
    amp = np.abs(np.fft.rfft(tr.data))        # FFT do sinal; abs() dá a amplitude de cada frequência
    freq = np.fft.rfftfreq(tr.stats.npts,     # frequência correspondente a cada valor da FFT...
                           d=tr.stats.delta)  # ...sabendo o intervalo entre amostras (0,01 s)
    return freq, amp                          # devolve as duas coisas

f_r, a_r = espectro(ruido)                    # espectro do ruído
f_s, a_s = espectro(sinal)                    # espectro do evento

plt.loglog(f_r, a_r, color="gray", lw=0.7, label="Ruído")    # eixos em escala logarítmica
plt.loglog(f_s, a_s, color="k", lw=0.7, label="Evento")
plt.xlabel("Frequência (Hz)")                 # rótulo do eixo x
plt.ylabel("Amplitude")                       # rótulo do eixo y
plt.title("Espectro de amplitude (FFT)")      # título
plt.xlim(0.1, 50)                             # mostra de 0,1 a 50 Hz
plt.legend()                                  # mostra a legenda (usa os "label")

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `np.fft.rfft(x)` | FFT para sinais reais: devolve números complexos, um por frequência | o vetor do sinal |
| `np.fft.rfftfreq(n, d)` | As frequências (Hz) de cada valor da FFT | `n`: nº de amostras; `d`: intervalo entre amostras (s) |
| `plt.loglog(x, y, ...)` | Gráfico com os dois eixos em escala logarítmica (bom para valores de ordens de grandeza diferentes) | `color`, `lw`, `label` |
| `plt.xlim(a, b)` | Limites do eixo x | — |
| `plt.legend()` | Mostra a legenda | — |

**Por que escala logarítmica?** As amplitudes variam de 10⁻⁸ a 10⁻³. Num eixo comum, tudo exceto os picos ficaria colado no zero.

**O que observar:** o evento tem muito mais energia que o ruído nas frequências de alguns Hz até dezenas de Hz. Em frequências baixas (abaixo de ~1 Hz) as duas curvas se aproximam.

A FFT pura é "serrilhada". A **PSD de Welch** suaviza: divide o sinal em pedaços sobrepostos, calcula o espectro de cada um e tira a média. O E3WS usa exatamente isso.

In [ ]:
from scipy.signal import welch                # função de PSD da biblioteca SciPy

fs = z.stats.sampling_rate                    # taxa de amostragem: 100 Hz

for tr_, nome, cor in [(ruido, "Ruído", "gray"), (sinal, "Evento", "k")]:   # repete para ruído e evento
    f, pxx = welch(tr_.data,                  # o sinal
                   fs=fs,                     # taxa de amostragem, para as frequências saírem em Hz
                   window="hann",             # janela de Hanning em cada pedaço
                   nperseg=512,               # tamanho de cada pedaço: 512 amostras (5,12 s)
                   noverlap=384,              # sobreposição entre pedaços: 384 amostras (75%)
                   nfft=512)                  # tamanho da FFT
    plt.semilogy(f, pxx, color=cor, label=nome)   # eixo y logarítmico, x linear
    print("%s: frequência dominante = %.1f Hz" % (nome, f[np.argmax(pxx)]))   # onde a PSD é máxima

plt.xlabel("Frequência (Hz)")
plt.ylabel("PSD ((m/s²)²/Hz)")
plt.title("PSD de Welch (mesmos parâmetros do E3WS)")
plt.xlim(0, 50)
plt.legend()

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `welch(x, ...)` | Estima a **PSD** (Densidade Espectral de Potência): energia por frequência | `fs`: taxa (Hz); `window`: formato da janela de cada pedaço; `nperseg`: tamanho de cada pedaço; `noverlap`: quantas amostras se repetem entre pedaços vizinhos; `nfft`: tamanho da FFT. Devolve `(frequências, potências)` |
| `plt.semilogy(x, y)` | Gráfico com só o eixo y logarítmico | — |
| `np.argmax(v)` | Posição do maior valor de um vetor | — |

**O que observar:** a frequência dominante do evento (onde ele tem mais energia) é um dos atributos espectrais do E3WS.

O **espectrograma** junta tempo e frequência: tempo no eixo x, frequência no eixo y, cor = energia.

In [ ]:
z.spectrogram(log=False,              # eixo de frequência linear
              wlen=2.0,               # cada coluna do gráfico usa 2 s de sinal
              per_lap=0.9,            # 90% de sobreposição entre colunas vizinhas (gráfico mais suave)
              dbscale=True,           # cores em decibéis (escala logarítmica de energia)
              cmap="viridis")         # paleta de cores

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `tr.spectrogram(...)` | Desenha o espectrograma | `log`: eixo de frequência logarítmico?; `wlen`: janela em s (maior = mais detalhe na frequência, menos no tempo); `per_lap`: sobreposição (0 a 1); `dbscale`: cores em dB; `cmap`: paleta |

**O que observar:** a faixa clara que surge de repente por volta de 33 s é a chegada da onda P, com energia numa faixa larga de frequências. Perto de 44 s aparece a onda S, mais concentrada em frequências mais baixas.

**Exercício 8:** teste `wlen=0.5` e `wlen=5`. O que fica mais nítido em cada caso, o tempo ou a frequência?

---
## Parte 9: filtros

**Conceito.** Um filtro deixa passar algumas frequências e corta outras.
- **Passa-baixa (`lowpass`):** mantém abaixo do corte.
- **Passa-alta (`highpass`):** mantém acima do corte.
- **Passa-banda (`bandpass`):** mantém entre dois cortes.

O E3WS usa dois passa-banda:
- **1–7 Hz** no detector e no picker: corta ruído de baixa frequência e terremotos muito distantes.
- **1–45 Hz** na estimativa da fonte: mantém o conteúdo de alta frequência, útil para magnitude e distância.

In [ ]:
base = ev.select(component="Z")[0].copy()     # vertical em m/s², copiado

versoes = {                                   # dicionário: nome -> sinal filtrado
    "Sem filtro": base.copy(),
    "Passa-baixa 1 Hz (o que o E3WS descarta)":
        base.copy().filter("lowpass", freq=1.0, zerophase=True),
    "Passa-banda 1–7 Hz (detector do E3WS)":
        base.copy().filter("bandpass", freqmin=1, freqmax=7, zerophase=True),
    "Passa-banda 1–45 Hz (fonte do E3WS)":
        base.copy().filter("bandpass", freqmin=1, freqmax=45, zerophase=True),
}

fig, ax = plt.subplots(len(versoes), 1, figsize=(12, 10), sharex=True)   # um painel por versão
for a, (nome, t) in zip(ax, versoes.items()): # percorre painéis e pares (nome, sinal)
    a.plot(t.times(), t.data, "k", lw=0.5)    # desenha o sinal
    a.set_title(nome, loc="left")             # título
    a.set_ylabel("m/s²")                      # unidade
ax[-1].set_xlabel("Tempo (s)")
plt.tight_layout()

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `{chave: valor, ...}` | Dicionário: guarda pares nome → valor | — |
| `dic.items()` | Percorre os pares (chave, valor) | — |
| `tr.filter(tipo, ...)` | Aplica um filtro Butterworth | `tipo`: `"lowpass"`, `"highpass"`, `"bandpass"`, `"bandstop"`; `freq`: corte (passa-baixa/alta); `freqmin`, `freqmax`: cortes do passa-banda; `corners`: "força" do corte (padrão 4; maior = corte mais abrupto); `zerophase` |
| `zerophase=True` | Filtra duas vezes (ida e volta) para **não deslocar o sinal no tempo**. Essencial para marcar a chegada da P no instante certo | — |

**O que observar:** compare a amplitude do passa-baixa de 1 Hz com a dos outros painéis. Em aceleração, quase toda a energia do evento está acima de 1 Hz.

**Exercício 9:** filtre de 10 a 30 Hz. O evento ainda aparece? E de 0,05 a 0,5 Hz?

---
## Parte 10: lendo o sismograma, ondas P e S

**Conceito.**
- **Onda P** (primária): chega primeiro e vibra na direção em que viaja. Para um evento que vem de baixo, aparece mais forte no **vertical (Z)**.
- **Onda S** (secundária): chega depois, vibra perpendicular à propagação. Aparece mais forte nos **horizontais (N, E)**.

In [ ]:
evf = ev.copy()                                            # cópia do evento em m/s²
evf.filter("bandpass", freqmin=1, freqmax=20, zerophase=True)   # passa-banda 1–20 Hz

t_P = UTCDateTime("2017-06-29T18:53:03.5")    # leitura aproximada da chegada da P
t_S = UTCDateTime("2017-06-29T18:53:14.6")    # leitura aproximada da chegada da S

fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)    # 3 painéis: Z, N, E
for a, comp in zip(ax, ["Z", "N", "E"]):                      # percorre painel e componente
    t = evf.select(component=comp)[0]                         # Trace do componente
    a.plot(t.times(), t.data, "k", lw=0.5)                    # sinal
    a.axvline(t_P - t.stats.starttime, color="b", label="P")  # linha vertical azul na P (em s desde o início)
    a.axvline(t_S - t.stats.starttime, color="r", label="S")  # linha vertical vermelha na S
    a.set_ylabel("HH" + comp)
ax[0].legend()                                                # legenda só no primeiro painel
ax[-1].set_xlabel("Tempo desde 18:52:30 (s)")
plt.tight_layout()

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `ax.axvline(x)` | Linha vertical na posição `x` | `color`, `label` |
| `t_P - t.stats.starttime` | Subtrair dois `UTCDateTime` dá a diferença em segundos | — |

**Estimando a distância pela diferença S − P.** As duas ondas saem juntas, mas a S é mais lenta. Quanto mais longe a fonte, maior a diferença. Com velocidades típicas da crosta (P ≈ 6 km/s, S ≈ 3,5 km/s):

$$d = (t_S - t_P) \cdot \frac{v_P \cdot v_S}{v_P - v_S} \approx (t_S - t_P) \times 8{,}4\ \text{km/s}$$

É a distância **hipocentral** (até o ponto de ruptura, em profundidade), e é grosseira. Mesmo assim, é a informação que faz o E3WS estimar melhor a distância quando a janela cresce e inclui a onda S.

In [ ]:
vp = 6.0                                      # velocidade típica da onda P na crosta (km/s)
vs = 3.5                                      # velocidade típica da onda S (km/s)
sp = t_S - t_P                                # diferença de chegada, em segundos
dist = sp * vp * vs / (vp - vs)               # fórmula acima
print("S - P = %.1f s  ->  distância aproximada = %.0f km" % (sp, dist))

**Exercício 10:** recorte de 18:53:00 a 18:53:20 e tente marcar P e S com mais precisão. Quanto a distância muda se você errar a S por 0,5 s?

---
## Parte 11: envelope

**Conceito.** O **envelope** é a "silhueta" da amplitude: liga os picos e ignora a oscilação rápida. É calculado pela transformada de Hilbert.

O E3WS calcula vários atributos do envelope. Segundo o artigo (Seção 5.2), o **centroide** e o **instante do máximo** estão entre os mais importantes para estimar a distância.

In [ ]:
from obspy.signal.filter import envelope       # função de envelope da ObsPy

zf = evf.select(component="Z")[0]             # vertical filtrado (1–20 Hz)
env = envelope(zf.data)                       # calcula o envelope (vetor do mesmo tamanho)
tempo = zf.times()                            # tempo de cada amostra (s)

plt.plot(tempo, zf.data, color="0.7", lw=0.5, label="Sinal (1–20 Hz)")   # sinal em cinza claro
plt.plot(tempo, env, "r", lw=1, label="Envelope")                        # envelope em vermelho
plt.xlabel("Tempo (s)")
plt.ylabel("m/s²")
plt.title("Componente Z e seu envelope")
plt.legend()

i_max = np.argmax(env)                        # posição do maior valor do envelope
centroide = np.sum(tempo * env) / np.sum(env) # "centro de massa" do envelope no tempo
print("Instante do máximo: %.2f s" % tempo[i_max])
print("Centroide no tempo: %.2f s" % centroide)

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `envelope(dados)` | Envelope pela transformada de Hilbert | o vetor do sinal |
| `color="0.7"` | Cor em tons de cinza: `"0"` = preto, `"1"` = branco | — |
| `np.sum(v)` | Soma dos elementos | — |
| `tempo * env` | Multiplica elemento a elemento (NumPy faz isso automaticamente) | — |

**O centroide** é a média dos tempos ponderada pela amplitude: se a energia se concentra no começo, o centroide fica cedo; se a coda (a "cauda" do sinal) é longa, ele vai mais para frente.

**Exercício 11:** calcule o envelope só da janela "7 s antes da P + 3 s depois" (a janela do E3WS). Onde fica o centroide?

---
## Parte 12: STA/LTA, o detector clássico

**Conceito.** O **STA/LTA** compara duas médias de energia:
- **STA** (*Short-Term Average*): média numa janela **curta** (1 s), que reage rápido.
- **LTA** (*Long-Term Average*): média numa janela **longa** (30 s), que representa o ruído de fundo.

Quando chega um evento, a STA sobe rápido e a LTA demora: a **razão STA/LTA** dispara. Passou de um limiar (ex.: 4) = início do disparo; caiu abaixo de outro (ex.: 1,5) = fim.

A razão não tem unidade, então dá no mesmo usar counts ou m/s².

In [ ]:
from obspy.signal.trigger import classic_sta_lta, plot_trigger, trigger_onset   # funções de detecção

zt = evento.select(component="Z")[0].copy()   # vertical em counts (para a razão, a unidade não importa)
zt.detrend("demean")                          # tira a média
zt.filter("bandpass", freqmin=1, freqmax=10, zerophase=True)   # foca na faixa do evento

fs = zt.stats.sampling_rate                   # 100 Hz
cft = classic_sta_lta(zt.data,                # calcula a razão STA/LTA para cada amostra
                      int(1 * fs),            # STA: 1 s = 100 amostras
                      int(30 * fs))           # LTA: 30 s = 3000 amostras
plot_trigger(zt, cft, 4.0, 1.5)               # desenha o sinal e a razão, com os disparos

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `classic_sta_lta(dados, nsta, nlta)` | Calcula a razão STA/LTA ao longo do sinal | `nsta`, `nlta`: tamanhos das janelas **em amostras** (por isso `segundos × fs`) |
| `int(x)` | Converte para inteiro | — |
| `plot_trigger(tr, cft, on, off)` | Desenha o sinal e a razão, marcando início (vermelho) e fim (azul) de cada disparo | `on`: limiar de início; `off`: limiar de fim |

**O que observar:** a razão dispara exatamente na chegada da P. Agora vamos rodar o mesmo detector no **dia inteiro**.

In [ ]:
dia = read(caminho("Z", 180))                 # dia inteiro em counts
dia.merge(fill_value=0)                       # um Trace só
tr_dia = dia[0]                               # pega o Trace
tr_dia.detrend("demean")                      # tira a média
tr_dia.filter("bandpass", freqmin=1, freqmax=10, zerophase=True)   # mesma faixa de antes

cft_dia = classic_sta_lta(tr_dia.data, int(1 * fs), int(30 * fs))   # razão STA/LTA do dia todo
disparos = trigger_onset(cft_dia, 4.0, 1.5)   # lista de disparos: [início, fim] em amostras
print("Disparos no dia 180:", len(disparos))  # quantos disparos

for ini, fim_ in disparos[:10]:               # mostra os 10 primeiros
    t_ini = tr_dia.stats.starttime + ini / fs # converte nº da amostra em instante de tempo
    dur = (fim_ - ini) / fs                   # duração em segundos
    print("  ", t_ini, " duração %.1f s" % dur)

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `trigger_onset(cft, on, off)` | Encontra os disparos na razão STA/LTA. Devolve uma lista de pares `[início, fim]` em **nº de amostras** | `on`, `off`: limiares |
| `len(lista)` | Quantidade de elementos | — |
| `for a, b in lista_de_pares:` | Percorre a lista "desempacotando" cada par em duas variáveis | — |

**O que observar:** são **centenas de disparos** num único dia. Alguns são terremotos; muitos são pequenos eventos locais, ruído humano, vento, ondas do mar. O STA/LTA só vê "aumento súbito de energia".

É esse limite que o E3WS ataca: em vez de um limiar fixo, um **classificador XGBoost** treinado para separar ruído, onda P e onda S a partir de 140 atributos.

**Exercício 12:** mude o limiar de 4,0 para 8,0. Quantos disparos sobram? Abra alguns com `read(..., starttime=, endtime=)` e tente dizer se são terremotos.

---
## Parte 13: três componentes e polarização

**Conceito.** Com três componentes dá para saber **em que direção** o solo se move. Na onda P, o movimento é quase uma linha reta apontando para a fonte (**polarização linear**). O ruído se move em direções variadas.

A matemática usa a **matriz de covariância** dos 3 componentes e seus **autovalores e autovetores**:
- **Maior autovalor:** intensidade do movimento na direção dominante.
- **Autovetor associado:** qual é essa direção (E, N, Z).
- **Razão maior autovalor / soma dos outros dois:** o quão "reto" é o movimento.

São os **5 atributos de polarização** do E3WS (`pb_utils_v16.py`, linhas 62 a 73), que alimentam a estimativa de back-azimuth.

**Um detalhe prático:** para combinar os componentes amostra a amostra, eles precisam ter o mesmo número de amostras. Veja o início de cada um:

In [ ]:
for t in evento:                                       # para cada componente...
    print(t.id, t.stats.starttime)                     # ...mostra o nome e o instante inicial
janela_teste = evento.slice(t_P - 7, t_P + 3)          # recorta 10 s
print("Amostras em cada componente:", [len(t) for t in janela_teste])   # conta as amostras de cada um

#### Funções desta célula

| Função | O que faz |
|---|---|
| `tr.id` | Nome completo do canal (`REDE.ESTACAO.LOCAL.CANAL`) |
| `[len(t) for t in st]` | *List comprehension*: monta uma lista aplicando `len` a cada Trace. É um `for` em uma linha |
| `len(tr)` | Nº de amostras do Trace |

**O que observar:** os relógios dos componentes diferem em **microssegundos** (…,005000 vs …,004999). Ao recortar, um componente pode ganhar uma amostra a mais. A função abaixo recorta e corta todos no menor tamanho.

In [ ]:
def recortar3(st, t1, t2):                    # função: recorta um Stream de 3 componentes
    r = st.slice(t1, t2).copy()               # recorta e copia
    n = min(len(t) for t in r)                # menor nº de amostras entre os componentes
    for t in r:                               # para cada componente...
        t.data = t.data[:n]                   # ...mantém só as n primeiras amostras
    return r                                  # devolve o Stream acertado

print([len(t) for t in recortar3(evento, t_P - 7, t_P + 3)])   # agora todos iguais

In [ ]:
def polarizacao(st3):                                   # recebe um Stream com E, N e Z
    dados = np.array([st3.select(component=c)[0].data   # monta uma matriz 3 x N:
                      for c in "ENZ"])                  # linha 1 = E, linha 2 = N, linha 3 = Z
    cov = np.cov(dados)                                 # matriz de covariância 3 x 3
    w, v = np.linalg.eig(cov)                           # autovalores (w) e autovetores (v, em colunas)
    ordenados = np.sort(w)                              # autovalores do menor para o maior
    vetor = v[:, np.argmax(w)]                          # autovetor (coluna) do maior autovalor
    razao = ordenados[-1] / (ordenados[-2] + ordenados[-3])   # maior ÷ (soma dos outros dois)
    return razao, vetor                                 # devolve a razão e a direção

janela_ruido = recortar3(evf, t_P - 10, t_P - 7)        # 3 s de ruído antes do evento
janela_P = recortar3(evf, t_P, t_P + 1)                 # primeiro 1 s de onda P

for nome, j in [("Ruído (3 s)", janela_ruido), ("Primeiro 1 s de P", janela_P)]:
    razao, vetor = polarizacao(j)                       # calcula para cada janela
    print("%-18s razão = %5.1f | autovetor (E, N, Z) = %s" % (nome, razao, np.round(vetor, 2)))

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `np.array(lista)` | Transforma uma lista em vetor/matriz NumPy | — |
| `np.cov(matriz)` | Matriz de covariância: cada linha da entrada é uma variável (E, N, Z) | — |
| `np.linalg.eig(m)` | Autovalores e autovetores. Os autovetores são as **colunas** de `v` | — |
| `np.sort(v)` | Ordena do menor para o maior | — |
| `v[:, k]` | A coluna `k` da matriz (`:` = todas as linhas) | — |
| `np.round(x, 2)` | Arredonda para 2 casas | — |
| `"%-18s"` | Texto alinhado à esquerda em 18 caracteres (deixa a saída em colunas) | — |

**O que observar:** compare a razão do ruído com a da onda P: quanto maior, mais "reto" o movimento. Na P, o autovetor tem uma componente Z grande (a onda vem de baixo), e as componentes E e N indicam a direção horizontal.

Uma estimativa grosseira da direção da fonte (back-azimuth) usa o ângulo entre E e N do autovetor. Ela tem uma **ambiguidade de 180°** (o autovetor pode apontar para frente ou para trás). O E3WS resolve isso com modelos treinados.

In [ ]:
razao, vetor = polarizacao(janela_P)                   # direção do primeiro 1 s de P
baz = np.degrees(np.arctan2(vetor[0], vetor[1])) % 360 # ângulo a partir do norte (E = x, N = y)
print("Direção estimada: %.0f° ou %.0f° (ambiguidade de 180°)" % (baz, (baz + 180) % 360))

#### Funções desta célula

| Função | O que faz |
|---|---|
| `np.arctan2(y, x)` | Ângulo (em radianos) do ponto (x, y). Aqui usamos `arctan2(E, N)` para medir a partir do norte, no sentido horário, como na bússola |
| `np.degrees(r)` | Radianos → graus |
| `% 360` | Resto da divisão por 360: põe o ângulo entre 0° e 360° |

**Exercício 13:** calcule a razão de polarização para a janela da onda S (`recortar3(evf, t_S, t_S + 1)`). É maior ou menor que a da P? Por quê?

---
## Parte 14: os 140 atributos do E3WS

**Conceito.** A função `st_FV` do E3WS recebe um Stream de 3 componentes (em counts) e devolve o **vetor de 140 atributos**. Internamente, ela faz o que vimos:

1. Tira média e tendência, aplica taper (Parte 6).
2. Filtra em passa-banda (Parte 9).
3. **Remove a resposta instrumental** e converte para m/s² (Parte 7). Para isso ela usa um arquivo de **polos e zeros** (`.pz`) em vez do StationXML.
4. Calcula os 5 atributos de polarização (Parte 13).
5. Para cada componente: 17 atributos temporais (incluindo os do envelope, Parte 11), 15 espectrais (PSD de Welch, Parte 8) e 13 MFCCs.

**Passo 1:** gerar um `.pz` só com a época de 2017. O arquivo `pb_ISPT.pz` da pasta tem as 96 épocas, e a função do E3WS leria a primeira (de 2008, outro sensor). Como os 3 componentes têm a mesma resposta, basta o vertical.

In [ ]:
import os                                               # biblioteca para lidar com caminhos de arquivos

pz_2017 = os.path.abspath("ISPT_HH_2017.pz")            # arquivo novo, na pasta deste notebook
inv_hhz = inv.select(channel="HHZ", time=t_evento)      # só o vertical, época de 2017
inv_hhz.write(pz_2017, format="SACPZ")                  # grava no formato de polos e zeros (SAC)
print(open(pz_2017).read())                             # mostra o conteúdo do arquivo

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `os.path.abspath(nome)` | Caminho completo de um arquivo a partir da pasta atual | — |
| `inv.write(arquivo, format=)` | Grava o Inventory em arquivo | `format`: `"STATIONXML"`, `"SACPZ"`, `"KML"`... |
| `open(arquivo).read()` | Abre um arquivo de texto e lê tudo | — |

**O que observar:** `ZEROS`, `POLES` e `CONSTANT` descrevem a curva da Parte 7 em forma de fórmula. Os 3 zeros em 0 indicam que o formato SAC descreve a resposta em deslocamento; o E3WS remove 2 deles para obter aceleração (`pb_utils_v16.py`, linha 47).

**Passo 2:** importar as funções do E3WS e calcular os atributos na janela que os regressores usam: **7 s antes da P + 3 s depois**, filtrada de 1 a 45 Hz.

In [ ]:
import sys                                                   # biblioteca do sistema do Python
sys.path.insert(0, os.path.abspath(os.path.join("..", "real_time")))   # ensina o Python a achar as funções do E3WS
import functions.pb_utils_v16 as pb_utils                    # funções de atributos do E3WS
from functions.utils import pb_SAM                           # classe que os modelos precisam para carregar

janela = recortar3(evento, t_P - 7, t_P + 3)  # janela de 10 s em counts, mesmo tamanho nos 3 componentes
janela.sort(keys=["channel"])                 # ordena E, N, Z (mesma ordem do script do E3WS)

FV = pb_utils.st_FV(janela,                   # o Stream de 3 componentes
                    pb_inst=True,             # True = remover a resposta instrumental
                    pzfile=pz_2017,           # arquivo de polos e zeros da ISPT (2017)
                    fmin=1.0,                 # filtro passa-banda: corte inferior (Hz)
                    fmax=45.0)                # corte superior (Hz)
FV = np.real(FV)                              # garante números reais (alguns cálculos geram parte imaginária nula)
print("Tamanho do vetor:", len(FV))
print("5 primeiros (polarização):", FV[:5])

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `sys.path.insert(0, pasta)` | Acrescenta uma pasta onde o Python procura módulos para `import` | `0`: no início da lista (prioridade máxima) |
| `os.path.join("..", "real_time")` | Monta um caminho. `".."` = pasta "de cima" | — |
| `st.sort(keys=[...])` | Ordena os Traces | `keys`: campos usados na ordenação |
| `pb_utils.st_FV(...)` | **Função do E3WS**: pré-processa e devolve os 140 atributos | `pb_inst`: remover resposta?; `pzfile`: arquivo `.pz`; `fmin`, `fmax`: passa-banda (o E3WS usa 1–7 Hz no detector e picker, 1–45 Hz nos regressores) |
| `np.real(x)` | Parte real de números complexos | — |

**O que observar:** o vetor tem **140 números**. É isso, e só isso, que os modelos do E3WS "veem". Cada janela de sinal vira uma linha de uma tabela com 140 colunas.

**Exercício 14:** calcule o vetor de uma janela de **ruído** (`recortar3(evento, t_P - 20, t_P - 10)`) e compare os 5 primeiros números com os do evento.

---
## Parte 15: primeira estimativa do E3WS na ISPT

**Conceito.** Agora aplicamos os modelos treinados do E3WS ao vetor de atributos. Repetimos com 3, 4, 5, 6 e 7 s de onda P, como o sistema faz em tempo real, a cada segundo.

**O que estamos pulando:** o **detector** e o **picker**. Nós fornecemos o instante da P (`t_P`) manualmente. O detector do repositório foi treinado para a estação SLRZ (Peru), e o da ISPT ainda precisa ser treinado com o ruído local (`models/DET/build_DET/`).

In [ ]:
import joblib                                 # biblioteca que carrega modelos salvos em disco

PASTA_MODELOS = os.path.join("..", "models")  # pasta dos modelos do E3WS

def caminho_modelo(alvo, segundos):           # monta o nome do arquivo do modelo
    nome = f"{alvo}_Whole_StackingXGB_7tp{segundos}f45_v16.0.joblib"   # ex.: MAG_..._7tp3f45_...
    return os.path.join(PASTA_MODELOS, alvo, nome)

resultados = []                               # lista vazia para guardar os resultados
for s in [3, 4, 5, 6, 7]:                     # segundos de onda P disponíveis
    j = recortar3(evento, t_P - 7, t_P + s)   # janela: 7 s antes da P até s segundos depois
    j.sort(keys=["channel"])                  # ordem E, N, Z
    fv = np.real([pb_utils.st_FV(j, True, pz_2017, 1.0, 45.0)])   # atributos; [ ] = tabela com 1 linha

    mag = joblib.load(caminho_modelo("MAG", s)).predict(fv)[0]    # carrega o modelo de magnitude e estima
    dis = joblib.load(caminho_modelo("DIS", s)).predict(fv)[0]    # distância epicentral (km)
    dep = joblib.load(caminho_modelo("DEP", s)).predict(fv)[0]    # profundidade (km)
    resultados.append((s, mag, dis, dep))     # guarda a linha de resultados
    print("%d s de P -> magnitude %.2f | distância %.0f km | profundidade %.0f km" % (s, mag, dis, dep))

#### Funções desta célula

| Função | O que faz | Parâmetros usados |
|---|---|---|
| `joblib.load(arquivo)` | Carrega um objeto Python salvo em disco (aqui, um modelo treinado: 10 XGBoost + LASSO) | caminho do `.joblib` |
| `modelo.predict(tabela)` | Aplica o modelo. A entrada é uma **tabela** (uma linha por janela, 140 colunas); a saída é um valor por linha | `fv` com formato (1, 140) |
| `[0]` | Pega o primeiro (e único) resultado | — |
| `lista.append(x)` | Acrescenta `x` ao fim da lista | — |

**Atenção:** os modelos são carregados dentro do laço só para deixar o código simples. Num sistema real, você carrega uma vez e reusa (o script `E3WS_rt.py` faz isso).

**O que observar**
- A **distância** estimada fica perto de 100 km. Compare com a estimativa por S − P da Parte 10: dois métodos independentes concordando é um bom sinal.
- A **magnitude** começa em ~4,0 com 3 s e se estabiliza em ~3,1 a partir de 4 s. É o "estimar cedo, corrigir continuamente" do artigo. A primeira estimativa é a mais incerta (no artigo, o erro típico com 3 s é de 0,2 a 0,4 unidades).

**Agora o back-azimuth.** Só temos o modelo de 3 s (cosseno e seno separados, como explica a Seção 3.3 do artigo).

In [ ]:
modelo_cos = joblib.load(os.path.join(PASTA_MODELOS, "BAZ",
                         "welloriented_BAZ_Cos_STEAD_StackingXGB_7tp3f45_v16.0.joblib"))   # modelo do cosseno
modelo_sen = joblib.load(os.path.join(PASTA_MODELOS, "BAZ",
                         "welloriented_BAZ_Sin_STEAD_StackingXGB_7tp3f45_v16.0.joblib"))   # modelo do seno

j3 = recortar3(evento, t_P - 7, t_P + 3)      # janela de 3 s de P
j3.sort(keys=["channel"])
fv3 = np.real([pb_utils.st_FV(j3, True, pz_2017, 1.0, 45.0)])   # atributos

cos_ = modelo_cos.predict(fv3)[0]             # cosseno estimado do ângulo
sen_ = modelo_sen.predict(fv3)[0]             # seno estimado do ângulo
baz_e3ws = np.degrees(np.arctan2(sen_, cos_)) % 360   # reconstrói o ângulo a partir de seno e cosseno
print("Back-azimuth do E3WS: %.0f°" % baz_e3ws)

#### Funções desta célula

| Função | O que faz |
|---|---|
| `np.arctan2(seno, cosseno)` | Reconstrói o ângulo a partir do seno e do cosseno, sem ambiguidade de quadrante |

**O que observar:** compare com a estimativa grosseira da Parte 13. Neste evento elas **não concordam** (cerca de 117° ou 297° pela polarização contra ~214° do E3WS). Sem o catálogo não dá para saber qual está mais perto da verdade. Lembre que o back-azimuth é o ponto mais fraco do E3WS segundo os próprios autores (erro médio de ~45° no artigo).

### Cuidado ao interpretar
- **Não temos o catálogo** deste evento para conferir. Ele não aparece nos catálogos globais USGS e ISC (eventos pequenos costumam ficar só no catálogo da rede local, o IG-EPN).
- A ISPT fica numa **ilha**, e a orientação dos horizontais e o tipo de ruído afetam o back-azimuth.
- O E3WS foi treinado para **M > 3**. Se este evento estiver perto de M3, está na borda do domínio de validade.

---
## Resumo

| Você aprendeu | Funções principais | Onde aparece no E3WS |
|---|---|---|
| Ler dados e metadados | `read`, `Stream`, `Trace`, `stats` | Leitura dos `.mseed` |
| Tratar falhas | `print_gaps`, `merge` | Controle de qualidade |
| Recortar janelas | `read(starttime, endtime)`, `slice`, `trim` | Janelas de 10 s e 7 s + 3 s |
| Limpar o sinal | `detrend`, `taper`, `resample` | Início da `st_FV` |
| Resposta instrumental | `read_inventory`, `select`, `remove_response` | Conversão para m/s² |
| Frequência | `np.fft.rfft`, `welch`, `spectrogram` | Atributos espectrais |
| Filtrar | `filter("bandpass", ...)` | 1–7 Hz e 1–45 Hz |
| P, S e distância | leitura visual, S − P | Picker e regressor de distância |
| Envelope | `envelope` | Atributos temporais |
| Detectar | `classic_sta_lta`, `trigger_onset` | O que o detector ML substitui |
| Polarização | `np.cov`, `np.linalg.eig` | Back-azimuth |
| Atributos e modelos | `st_FV`, `joblib.load`, `predict` | O pipeline completo |

### Perguntas para levar ao Pablo
1. Os arquivos de 2017 dizem localização `00`, mas o StationXML diz vazio para a época de 2017. Posso assumir que é o mesmo sensor (Trillium 120PA + RT-130)?
2. Existe catálogo do IG-EPN para 2017 (tempo, magnitude, localização) para conferir as estimativas?
3. Os horizontais da ISPT estão bem orientados para norte e leste? Isso afeta o back-azimuth.
4. Próximo passo: treinar o detector da ISPT com o ruído local (`models/DET/build_DET/`)?